# Backtest Integrity Guard — 2-minute quickstart

This notebook installs the public PyPI package and demonstrates both a passing audit and a deliberately broken 5-minute OHLCV series.


In [ ]:
%pip -q install backtest-integrity-guard


In [ ]:
from pathlib import Path
import subprocess, textwrap

Path('clean.csv').write_text(textwrap.dedent('''\
timestamp,open,high,low,close,volume
2026-01-01T00:00:00Z,100,102,99,101,1200
2026-01-01T00:05:00Z,101,103,100,102,1400
'''))
subprocess.run(['btguard','ohlcv','clean.csv','--hash-input'], check=True)


Now remove two expected 5-minute bars. `btguard` should return a non-zero exit code and report `MISSING_BARS`.


In [ ]:
Path('broken.csv').write_text(textwrap.dedent('''\
timestamp,open,high,low,close,volume
2026-01-01T00:00:00Z,100,102,99,101,1200
2026-01-01T00:15:00Z,101,103,100,102,1400
'''))
result = subprocess.run(['btguard','ohlcv','broken.csv','--interval-seconds','300'], text=True, capture_output=True)
print(result.stdout)
print('exit code:', result.returncode)


The same CLI can audit signal-to-execution causality, same-bar stop/target ambiguity, and SHA256-frozen inputs. See the repository README for the complete command set.
